In [ ]:
import polars as pl
from haystack import Document
from haystack.components.evaluators import DocumentNDCGEvaluator, DocumentMAPEvaluator, DocumentRecallEvaluator
from haystack.components.evaluators.document_recall import RecallMode

from app.word2vec.pipeline import create as create_w2v
from app.sbert.pipeline import create as create_sbert
from app.llm.pipeline import create as create_llm

In [ ]:
datasets = pl.read_csv("./data/50-fyp.csv") \
    .filter(pl.col("abstract").is_not_null()) \
    .unique("title")
datasets

In [ ]:
documents = [Document(id=row["title"], content=row["abstract"]) for row in datasets.iter_rows(named=True)]

In [ ]:
pipeline = (create_w2v(documents, force_embeddings=False)
            # .with_hyde()
            # .with_bm25()
            # .with_evaluator()
            .make())

In [ ]:
query = "Design of automated and robotic systems"
top_k = 10
results = pipeline.run(data={
    "query": {"query": query, "top_k": top_k}
}, include_outputs_from={"result", "evaluator"})
results

In [ ]:
# Query : Design of automated and robotic system
ground_truth = datasets.filter(pl.col("title").is_in((
    "Robot Control Interface with Video Feedback for Wireless Camera",
    "Design and Build Delivery Robot",
    "Arduino Based Dual-Axis Solar Tracking System",
    "Identification in the Presence of Speed Bumps through Few-Shot Learning",
    "Robot Control Interface with Video Feedback for Wireless Camera",
    "Single-Sample Face Recognition for Attendance Record",
    "Cable Fault Recognition Using Image Analysis Of Phase Resolved Partial Discharge Pattern",
    "Banana Leaf Disease Detection Using Image Processing Methods",
    "Design of an Automated Powder Spray System for Cable Trunk",
    "Design and Develop a Cost-Effective Hydroponics Autodoser System",
)))
ground_truth

In [ ]:
# # Query : Biological diversity and biochemical analysis of natural substances
# ground_truth = datasets.filter(pl.col("title").is_in((
#     "Extraction Yield and Antimicrobial of Banana (Musa) Peel, Calamansi Lime (Citrus X Microcarpa) Peel and Durian (Durio Zibethinus) Husk Extracted by Using Different Solvents",
#     "Evaluation of Antihyperglycaemic and Antimicrobial Effects of Different Extracts of Andrographis paniculata and Herbal Tea",
#     "Interaction of Apigenin and Hesperetin on Xanthine Oxidase Inhibition and Their Inhibitory Mechanism",
#     "A Review on the Effect of Different Yeasts on the Fermentation of Red Dragon Fruit",
#     "Extraction Yield and Antimicrobial of Banana (Musa) Peel, Calamansi Lime (Citrus X Microcarpa) Peel and Durian (Durio Zibethinus) Husk Extracted by Using Different Solvents",
#     "Bioactive Compounds and Physical Analysis of Bentong Gingers and Commercial Gingers Powder by Different Storage Temperature",
#     "Effect of Lactic Acid Bacteria on the Physicochemical, Anti-Inflammatory Antioxidant and Microbiological Properties of Kombucha",
# )))
# ground_truth

In [ ]:
# # Query :
# ground_truth = datasets.filter(pl.col("title").is_in((
# )))
# ground_truth

In [ ]:
retrieved_documents = results["result"]["documents"]
ground_truth_documents = [Document(id=row["title"], content=row["abstract"]) for row in ground_truth.iter_rows(named=True)]
relevant_documents = [doc for doc in results["result"]["documents"] if any(gt.id == doc.id for gt in ground_truth_documents)]

In [ ]:
precision = len(relevant_documents) / len(retrieved_documents)
precision

In [ ]:
recall = len(relevant_documents) / len(ground_truth_documents)
recall

In [ ]:
fallout = (len(retrieved_documents) - len(relevant_documents)) / (len(documents) - len(ground_truth_documents))
fallout

In [ ]:
f_b = lambda b: (1 + b ** 2) * (precision * recall) / (b ** 2 * precision + recall)
f_b(1)

In [ ]:
f_b(2)

In [ ]:
recall = DocumentRecallEvaluator(mode=RecallMode.MULTI_HIT)
recall_result = recall.run(
    ground_truth_documents=[ground_truth_documents],
    retrieved_documents=[results["result"]["documents"]]
)
recall_result

In [ ]:
ndcg_eval = DocumentNDCGEvaluator()
ndcg_result = ndcg_eval.run(
    ground_truth_documents=[ground_truth_documents],
    retrieved_documents=[results["result"]["documents"]]
)
ndcg_result

In [ ]:
map_eval = DocumentMAPEvaluator()
map_result = map_eval.run(
    ground_truth_documents=[ground_truth_documents],
    retrieved_documents=[results["result"]["documents"]]
)
map_result